In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import StandardScaler
import copy
import seaborn as sns
import tensorflow as tf
from sklearn.linear_model import LinearRegression

In [ ]:
#read the file to see what columns it actually has
df = pd.read_csv("sample_data/SeoulBikeData.csv", encoding='latin1')
print("Actual columns in the dataset:")
print(df.columns.tolist())
print(f"\nNumber of columns: {len(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

In [ ]:
dataset_cols = ["bike_count", "hour", "temp", "humidity", "wind", "visibility", "dew_pt_temp", "solar_radiation", "rainfall", "snow", "functional"]

## create data file
df = pd.read_csv("sample_data/SeoulBikeData.csv", encoding='latin')
df = df.drop(["Date", "Seasons", "Holiday"], axis=1)
df.head()

In [ ]:
df.columns = dataset_cols
df.head()
df["functional"] = (df["functional"] == "Yes").astype(int)
df.head()

## change hour to 12
df = df[df["hour"] == 12]
df.head()

## drop hour column
df = df.drop(["hour"], axis=1)
df.head()

In [ ]:
## plotting
for label in df.columns[1:]:
  plt.scatter(df[label], df["bike_count"])
  plt.title(label)
  plt.ylabel("Hike count at midday")
  plt.xlabel(label)
  plt.show()

## dropping wind visibility and functional
df = df.drop(["wind", "visibility", "functional"], axis=1)
df.head()


In [ ]:
## split into training, validation, test dataset
train, valid, test = np.split(df.sample(frac=1), [int(0.6*len(df)), int(0.8*len(df))])

In [ ]:
## function to get x and y
def get_xy(dataframe, y_label, x_labels=None):
  dataframe = copy.deepcopy(dataframe)
  if x_labels is None:
    X = dataframe[[c for c in dataframe.columns if c!=y_label]].values
  else:
    if len(x_labels) == 1:
      X = dataframe[x_labels[0]].values.reshape(-1, 1)
    else:
      X = dataframe[x_labels].values

  y = dataframe[y_label].values.reshape(-1, 1)
  data = np.hstack((X, y))

  return data, X, y


_, X_train_temp, y_train_temp = get_xy(train, "bike_count", x_labels=["temp"] )
_, X_valid_temp, y_valid_temp = get_xy(valid, "bike_count", x_labels=["temp"] )
_, X_test_temp, y_test_temp = get_xy(test, "bike_count", x_labels=["temp"] )


In [ ]:
## create regressor model
temp_reg = LinearRegression()
temp_reg.fit(X_train_temp, y_train_temp)



In [ ]:
print(temp_reg.coef_, temp_reg.intercept_)

temp_reg.score(X_test_temp, y_test_temp)

In [ ]:
plt.scatter(X_train_temp, y_train_temp, label="Data", color="blue")
x = tf.linspace(-20, 40, 100)

plt.plot(x, temp_reg.predict(np.array(x).reshape(-1, 1)), label="Fit", color="red", linewidth=3)
plt.legend()
plt.title("Bikes vs Temp")
plt.ylabel("No. of Bikes")
plt.xlabel("Temp")
plt.show()

In [ ]:
## Multiple Linear Regression

## split into training, validation, test dataset
train, valid, test = np.split(df.sample(frac=1), [int(0.6*len(df)), int(0.8*len(df))])
_, X_train_all, y_train_all = get_xy(train, "bike_count", x_labels=df.columns[1:].tolist())
_, X_valid_all, y_valid_all = get_xy(valid, "bike_count", x_labels=df.columns[1:].tolist())
_, X_test_all, y_test_all = get_xy(test, "bike_count", x_labels=df.columns[1:].tolist())


In [ ]:
all_reg = LinearRegression()
all_reg.fit(X_train_all, y_train_all)

all_reg.score(X_test_all, y_test_all)

In [ ]:
def plot_loss(history):
  plt.plot(history.history["loss"], label="loss")
  plt.plot(history.history["val_loss"], label="val_loss")
  plt.xlabel("Epoch")
  plt.ylabel("MSE")
  plt.legend()
  plt.grid(True)
  plt.show()

In [ ]:
## Regression with Neural Net
# build model

temp_normalizer = tf.keras.layers.Normalization(input_shape=(1,), axis=None)
temp_normalizer.adapt(X_train_temp.reshape(-1))

temp_nnet_model = tf.keras.Sequential([
    temp_normalizer,
    tf.keras.layers.Dense(1)
])


In [ ]:
temp_nnet_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.1), loss='mean_squared_error')

In [ ]:
history = temp_nnet_model.fit(
    X_train_temp.reshape(-1), y_train_temp,
    verbose=0,
    epochs=1000,
    validation_data=(X_valid_temp, y_valid_temp)
)

In [ ]:
plot_loss(history)

In [ ]:
plt.scatter(X_train_temp, y_train_temp, label="Data", color="blue")
x = tf.linspace(-20, 40, 100)

plt.plot(x, temp_nnet_model.predict(np.array(x).reshape(-1, 1)), label="Fit", color="red", linewidth=3)
plt.legend()
plt.title("Bikes vs Temp")
plt.ylabel("No. of Bikes")
plt.xlabel("Temp")
plt.show()

In [ ]:
## Neural Net
temp_normalizer = tf.keras.layers.Normalization(input_shape=(1,), axis=None)
temp_normalizer.adapt(X_train_temp.reshape(-1))

nnet_model = tf.keras.Sequential([
    temp_normalizer,
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    #tf.keras.layers.Dense(1, activation="relu")
    tf.keras.layers.Dense(1)

])

nnet_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mean_squared_error')


In [ ]:
## training
history = nnet_model.fit(
    X_train_temp, y_train_temp,
    validation_data=(X_valid_temp, y_valid_temp),
    verbose=0,
    epochs=100
)

In [ ]:
plot_loss(history)

In [ ]:
plt.scatter(X_train_temp, y_train_temp, label="Data", color="blue")
x = tf.linspace(-20, 40, 100)

plt.plot(x,nnet_model.predict(np.array(x).reshape(-1, 1)), label="Fit", color="red", linewidth=3)
plt.legend()
plt.title("Bikes vs Temp")
plt.ylabel("No. of Bikes")
plt.xlabel("Temp")
plt.show()

In [ ]:
## Neural Net
## with multiple inputs
all_normalizer = tf.keras.layers.Normalization(input_shape=(6,), axis=-1)
all_normalizer.adapt(X_train_all)

In [ ]:
## Neural Net

nnet_model = tf.keras.Sequential([
    all_normalizer,
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation="relu")
    #tf.keras.layers.Dense(1)

])

nnet_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mean_squared_error')


In [ ]:
history = nnet_model.fit(
    X_train_all, y_train_all,
    validation_data=(X_valid_all, y_valid_all),
    verbose=0,
    epochs=100
)

In [ ]:
plot_loss(history)

In [ ]:
## calculate MSE for linear regressor and neural net
y_pred_lr = all_reg.predict(X_test_all)
y_pred_nnet = nnet_model(X_test_all)

In [ ]:
def MSE(y_pred, yreal):
  return (np.square(y_pred - y real)).mean

MSe(y_pred_lr, y_test_all)